# LIBERO Simulation Benchmark — Dependency Setup

Installs everything needed to run `benchmark_libero.py` against a trained `TransformerFlowMatchingPolicy`.

**Target**: Google Colab (Linux) with GPU, headless rendering via Xvfb + EGL.

**Order matters**:
1. Pin `numpy<2.0` first (robosuite 1.4.x + numba require it).
2. Install system libs for headless rendering.
3. Clone the `lerobot-piper` repo so `benchmark_libero.py` and the policy code are on disk.
4. Install LIBERO from source with `--no-deps` so it does not downgrade numpy to a broken 1.22.
5. Install the remaining simulation and policy dependencies.
6. Restart the kernel once, then verify imports.
7. (Optional) Smoke-test rendering.
8. Run the full benchmark.

## 1. Pin NumPy < 2.0

robosuite 1.4.1 and numba pin to `numpy<2`. Doing this first prevents later installs from picking up an incompatible 2.x wheel.

In [ ]:
!pip install "numpy<2.0"

## 2. System Dependencies (Headless Rendering)

Required when running without a physical display (Colab, remote servers). `MUJOCO_GL=egl` gives GPU-accelerated offscreen rendering.

In [ ]:
!apt-get install -y xvfb libgl1-mesa-glx libegl1-mesa libgles2-mesa-dev 2>/dev/null

## 3. Clone the `lerobot-piper` Repo

`benchmark_libero.py` lives in this repo and imports `models.transformer_flow_matching` from `src/`. Cloning here makes both the script and the policy class available.

In [ ]:
import os
if not os.path.exists('lerobot-piper'):
    !git clone https://github.com/mayuanyang/lerobot-piper.git

!ls lerobot-piper/src/benchmark_libero.py

## 4. Install LIBERO (from source, no deps)

`--no-deps` is critical — LIBERO's `setup.py` pins `numpy==1.22.4` which is broken on modern Python. We install its real dependencies in the next cell.

In [ ]:
import os
if not os.path.exists('LIBERO'):
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git

!pip install -e ./LIBERO --no-deps
!pip install hydra-core==1.2.0

## 5. Simulation Dependencies

Versions match what `benchmark_libero.py` is tested against. `robosuite==1.4.1` is required — newer 1.5.x has API changes the LIBERO env loaders cannot handle, and `benchmark_libero.py` applies compatibility monkeypatches specifically for 1.4.1.

In [ ]:
!pip install bddl robosuite==1.4.1 mujoco
!pip install pyvirtualdisplay

## 6. Policy and Dataset Dependencies

`lerobot==0.4.0` provides `LeRobotDatasetMetadata` for normalization stats. `transformers` is needed by `TransformerFlowMatchingPolicy` (SmolVLM vision backbone).

In [ ]:
!pip install lerobot==0.4.0
!pip install transformers
!pip install decord opencv-python num2words

## 7. Restart Kernel

After the numpy downgrade and the new C-extension installs (mujoco, robosuite), a one-time kernel restart is required so already-imported modules pick up the correct binaries.

**Run the cell below — it will restart the kernel automatically. Then re-open the notebook and continue from the verification cell.**

In [ ]:
import os
os.kill(os.getpid(), 9)

## 8. Verify Installation

After restart, run this cell. It mirrors the imports `benchmark_libero.py` performs, applies the same robosuite monkeypatch, and confirms the policy class is importable from the cloned repo.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import pandas as pd
import huggingface_hub

assert int(np.__version__.split('.')[0]) < 2, f"numpy {np.__version__} — must be <2.0"

# Apply the same robosuite 1.4.1 compatibility patches benchmark_libero.py uses
import json
import robosuite.controllers as controllers
import robosuite.robots as robots
from robosuite.robots.robot import Robot

def _load_controller_config(custom_fpath=None, default_controller=None):
    if custom_fpath is not None:
        with open(custom_fpath) as f:
            return json.load(f)
    return {"type": default_controller} if default_controller else {}

if not hasattr(controllers, "load_controller_config"):
    setattr(controllers, "load_controller_config", _load_controller_config)
if not hasattr(robots, "SingleArm"):
    setattr(robots, "SingleArm", Robot)

from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
import mujoco
import pyvirtualdisplay

# Make the cloned lerobot-piper/src importable so we can load the policy class
repo_src = Path("lerobot-piper/src").resolve()
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from models.transformer_flow_matching.transformer_flow_matching_policy import (
    TransformerFlowMatchingPolicy,
)

print(f"numpy        : {np.__version__}")
print(f"torch        : {torch.__version__}  (cuda={torch.cuda.is_available()})")
print(f"mujoco       : {mujoco.__version__}")
print(f"LIBERO suites: {list(benchmark.get_benchmark_dict().keys())}")
print(f"Policy class : {TransformerFlowMatchingPolicy.__name__} importable from {repo_src}")
print("\nAll dependencies for benchmark_libero.py are installed.")

## 9. (Optional) Smoke-Test Headless Rendering

Boots a virtual display, builds one LIBERO env, and resets it. Confirms the EGL + Xvfb pipeline works before launching a long benchmark run.

In [ ]:
import os
from pyvirtualdisplay import Display

display = Display(visible=0, size=(1400, 900))
display.start()
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("DISPLAY", ":99")

task_suite = benchmark.get_benchmark_dict()["libero_spatial"]()
env = OffScreenRenderEnv(
    bddl_file_name=task_suite.get_task_bddl_file_path(0),
    camera_heights=256,
    camera_widths=256,
)
obs = env.reset()
env.close()

print("Headless render OK. Observation keys:")
for k, v in obs.items():
    arr = np.array(v)
    print(f"  {k:40s}  shape={arr.shape}  dtype={arr.dtype}")

## 10. Run the Full Benchmark

Runs `benchmark_libero.py` across all four LIBERO suites (`libero_spatial`, `libero_object`, `libero_goal`, `libero_long`).

**Paper-standard cost**: 50 rollouts × 10 tasks × 4 suites = 2000 rollouts. Expect several hours on a single GPU.

For a fast sanity check first, drop `NUM_ROLLOUTS` to 5 and `MAX_STEPS` to 150. Results are written to `OUTPUT_JSON` so you can re-load them without re-running.

In [ ]:
CHECKPOINT     = "ISdept/fm64-libero"   # HF repo ID or local checkpoint dir
NUM_ROLLOUTS   = 50                      # paper standard
MAX_STEPS      = 300
N_ACTION_STEPS = 8
IMAGE_SIZE     = 256
OUTPUT_JSON    = "libero_benchmark_results.json"

!python lerobot-piper/src/benchmark_libero.py \
    --checkpoint {CHECKPOINT} \
    --suites libero_spatial libero_object libero_goal libero_long \
    --num_rollouts {NUM_ROLLOUTS} \
    --max_steps {MAX_STEPS} \
    --n_action_steps {N_ACTION_STEPS} \
    --image_size {IMAGE_SIZE} \
    --output_json {OUTPUT_JSON} \
    --headless

In [ ]:
# Reload the saved JSON results into a DataFrame
import json
import pandas as pd

with open(OUTPUT_JSON) as f:
    all_results = json.load(f)

rows = []
for suite, data in all_results.items():
    for task_name, td in data["tasks"].items():
        rows.append({
            "Suite":    suite,
            "Task":     task_name,
            "Success":  f"{td['successes']}/{td['total']}",
            "Rate (%)": round(td["rate"], 1),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print("\nSuite averages:")
for suite, data in all_results.items():
    print(f"  {suite:20s}  {data['suite_avg']:5.1f}%")